
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [ ]:
# your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [ ]:
# mount the drive here

Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [ ]:
import pandas as pd

In [ ]:
# read the data here
# remember to print the data structure you produce and the number of stimuli it contains!

df_rawc = pd.read_csv("data/processed/raw-c.csv")

In [ ]:
df_rawc.head()

In [ ]:
print(df_rawc.shape)
print(df_rawc.dtypes)
df_rawc.info()

In [ ]:
df = df_rawc[["word", "string", "sentence1", "sentence2", "v1", "v2"]]
df = df.rename(columns={"word": "lemma", "string": "word"})

In [ ]:
df

In [ ]:
df_sentences = (
    pd.concat(
        [
            df[["lemma", "word", "sentence1", "v1"]].rename(
                columns={"sentence1": "sentence", "v1": "v"}
            ),
            df[["lemma", "word", "sentence2", "v2"]].rename(
                columns={"sentence2": "sentence", "v2": "v"}
            ),
        ]
    )
    .drop_duplicates(subset=["lemma", "word", "sentence"])
    .reset_index(drop=True)
)

In [50]:
df_sentences

,lemma,word,sentence,v,layer_4,layer_8,layer_12,static
0,act,act,It was a desperate act.,M1_a,"[-0.05827096, -0.27012187, 0.42561162, -0.1391...","[-0.16379866, 0.2822563, 0.25095946, 0.1332268...","[-0.07384763, 0.29591513, 0.20930925, 0.215899...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
1,act,act,It was a humane act.,M1_b,"[0.08830759, -0.44680747, 0.38992333, -0.44328...","[-0.20170821, -0.27367488, 0.0874715, -0.19021...","[0.17367683, -0.016312985, 0.080417745, 0.2597...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
2,act,act,It was a magic act.,M2_a,"[-0.2717343, -0.39731777, 0.7297973, -0.577845...","[-0.13248087, -0.059305325, 0.36231574, -0.200...","[0.06568544, 0.17123577, 0.07869129, 0.1753310...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
3,appeal,appeal,He had a universal appeal.,M1_a,"[0.91521156, 1.0002257, -1.0877806, 1.2197236,...","[1.040754, 1.0634533, -0.08483757, 0.69389874,...","[0.39856437, 0.5085523, 0.10799585, 0.30021843...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
4,appeal,appeal,He had an undefinable appeal.,M1_b,"[0.3740786, 0.4802913, -0.7315045, 0.95309365,...","[0.7130884, 0.82163036, -0.084937364, 0.935322...","[0.4132054, 0.4139912, 0.075453654, 0.31927937...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
...,...,...,...,...,...,...,...,...
443,title,title,She liked the professional title.,M2_b,"[-0.31480587, 0.32864916, 0.19436167, 0.727829...","[-0.36327946, 0.6725698, 0.288711, 0.83076096,...","[0.08188132, 0.005937787, 0.42133862, 0.253184...","[-0.040719163, -0.0012473409, -0.02736756, 0.0..."
444,toast,toasted,They toasted the bride.,M2_b,"[-0.1185866, -0.014790956, -0.30728626, 1.1917...","[-0.46605432, 0.24366164, -0.18896995, 0.39927...","[-0.25262544, -0.018994475, 0.26549026, -0.108...","[-0.024888199, 0.043743096, -0.042333048, 0.03..."
445,toll,toll,It was a bell toll.,M2_b,"[-1.2667145, 0.2958592, 0.33129865, 0.29203457...","[-1.302903, -0.5709334, -0.59906083, -0.050656...","[-0.33465603, -0.027967393, 0.25729212, 0.2672...","[-0.059510894, -0.007590637, 0.07930978, -0.06..."
446,track,tracks,She saw the snowshoe tracks.,M2_b,"[0.11191279, 0.18632755, -0.13256125, -0.48526...","[0.28136143, 0.053297315, 0.262446, 0.53461206...","[0.15401953, -0.15668522, 0.20108204, -0.15116...","[-0.062270977, -0.02131133, 0.06818508, -0.016..."


## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

In [ ]:
# install the psycho-embeddings package here

Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [ ]:
# your code here
from psycho_embeddings import ContextualizedEmbedder

Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [ ]:
# your code here

model = ContextualizedEmbedder("bert-base-cased", max_length=128)

In [ ]:
embeddings = model.embed(
    words=["assignment"],
    target_texts=["I am having so much fun with this assignment!"],
    layers_id=[4, 8, 12],
    batch_size=8,
    return_static=True,
)

In [ ]:
print(embeddings.keys())
print(len(embeddings))

The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [ ]:
# your code here


def get_embeddings(row):
    result = model.embed(
        words=[row["word"]],
        target_texts=[row["sentence"]],
        layers_id=[4, 8, 12],
        return_static=True,
    )
    return {f"layer_{k}" if k != -1 else "static": v[0] for k, v in result.items()}

In [ ]:
embedding_cols = df_sentences.apply(get_embeddings, axis=1, result_type="expand")
df_sentences = pd.concat([df_sentences, embedding_cols], axis=1)

In [78]:
df_sentences

,lemma,word,sentence,v,layer_4,layer_8,layer_12,static
0,act,act,It was a desperate act.,M1_a,"[-0.05827096, -0.27012187, 0.42561162, -0.1391...","[-0.16379866, 0.2822563, 0.25095946, 0.1332268...","[-0.07384763, 0.29591513, 0.20930925, 0.215899...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
1,act,act,It was a humane act.,M1_b,"[0.08830759, -0.44680747, 0.38992333, -0.44328...","[-0.20170821, -0.27367488, 0.0874715, -0.19021...","[0.17367683, -0.016312985, 0.080417745, 0.2597...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
2,act,act,It was a magic act.,M2_a,"[-0.2717343, -0.39731777, 0.7297973, -0.577845...","[-0.13248087, -0.059305325, 0.36231574, -0.200...","[0.06568544, 0.17123577, 0.07869129, 0.1753310...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
3,appeal,appeal,He had a universal appeal.,M1_a,"[0.91521156, 1.0002257, -1.0877806, 1.2197236,...","[1.040754, 1.0634533, -0.08483757, 0.69389874,...","[0.39856437, 0.5085523, 0.10799585, 0.30021843...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
4,appeal,appeal,He had an undefinable appeal.,M1_b,"[0.3740786, 0.4802913, -0.7315045, 0.95309365,...","[0.7130884, 0.82163036, -0.084937364, 0.935322...","[0.4132054, 0.4139912, 0.075453654, 0.31927937...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
...,...,...,...,...,...,...,...,...
443,title,title,She liked the professional title.,M2_b,"[-0.31480587, 0.32864916, 0.19436167, 0.727829...","[-0.36327946, 0.6725698, 0.288711, 0.83076096,...","[0.08188132, 0.005937787, 0.42133862, 0.253184...","[-0.040719163, -0.0012473409, -0.02736756, 0.0..."
444,toast,toasted,They toasted the bride.,M2_b,"[-0.1185866, -0.014790956, -0.30728626, 1.1917...","[-0.46605432, 0.24366164, -0.18896995, 0.39927...","[-0.25262544, -0.018994475, 0.26549026, -0.108...","[-0.024888199, 0.043743096, -0.042333048, 0.03..."
445,toll,toll,It was a bell toll.,M2_b,"[-1.2667145, 0.2958592, 0.33129865, 0.29203457...","[-1.302903, -0.5709334, -0.59906083, -0.050656...","[-0.33465603, -0.027967393, 0.25729212, 0.2672...","[-0.059510894, -0.007590637, 0.07930978, -0.06..."
446,track,tracks,She saw the snowshoe tracks.,M2_b,"[0.11191279, 0.18632755, -0.13256125, -0.48526...","[0.28136143, 0.053297315, 0.262446, 0.53461206...","[0.15401953, -0.15668522, 0.20108204, -0.15116...","[-0.062270977, -0.02131133, 0.06818508, -0.016..."


In [ ]:
df_sentences.to_pickle("data/processed/with_embs.pkl")

# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [ ]:
# your code here
import nltk

nltk.download("wordnet")
from nltk.corpus import wordnet as wn

In [ ]:
def get_senses_and_glosses(word):
    """
    Returns a dict of {sense_name: gloss} for synsets that contain
    the word or derivatives thereof.
    """
    senses = {}
    for synset in wn.synsets(word):
        # filter to only lemmas that match the word
        matching_lemmas = [l for l in synset.lemmas() if l.name().lower().startswith(word.lower())]
        if matching_lemmas:
            senses[synset.name()] = synset.definition()
    return senses

In [ ]:
bat_senses = get_senses_and_glosses("bat")
for sense, gloss in bat_senses.items():
    print(f"{sense}: {gloss}")

## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [73]:
def retrieve_sense_embeddings(
    lemma_list: str | list[str], 
    get_sense_and_glosses_func, 
    LAST_LAYER_IDX: int = 12
) -> pd.DataFrame:
    sense_embeddings = {"lemma": [], "sense": [], "gloss": [], "embedding": []}

    if isinstance(lemma_list, str):
        lemma_list = [lemma_list]
        
    for lemma in lemma_list:
        for sense, gloss in get_sense_and_glosses_func(lemma).items():
            trimmed_sense = sense.split(".")[0]
            contextualized_gloss = f"{trimmed_sense}: {gloss}"
            emb = model.embed(words=[trimmed_sense], target_texts=[contextualized_gloss], layers_id=[LAST_LAYER_IDX], show_progress=False)
            sense_embeddings["lemma"].append(lemma)
            sense_embeddings["sense"].append(trimmed_sense)
            sense_embeddings["gloss"].append(gloss)
            sense_embeddings["embedding"].append(emb[LAST_LAYER_IDX])
    
    sense_embeddings_df = pd.DataFrame.from_dict(sense_embeddings)

    return sense_embeddings_df

In [74]:
output_bank = retrieve_sense_embeddings(
    "bank",
    get_senses_and_glosses
)

output_bank

Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 66.52 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 326.58 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 424.10 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned mem

,lemma,sense,gloss,embedding
0,bank,bank,sloping land (especially the slope beside a bo...,"[[0.19834378, -0.30834326, 0.015301091, -0.158..."
1,bank,depository_financial_institution,a financial institution that accepts deposits ...,"[[-0.4994915, -0.053576164, 0.18395343, -0.058..."
2,bank,bank,a long ridge or pile,"[[-0.020879323, 0.34919986, -0.10445713, 0.075..."
3,bank,bank,an arrangement of similar objects in a row or ...,"[[0.122795455, 0.07401531, 0.1653089, -0.09428..."
4,bank,bank,a supply or stock held in reserve for future u...,"[[-0.52822095, 0.034207292, 0.040675588, -0.27..."
5,bank,bank,the funds held by a gambling house or the deal...,"[[-0.4247744, 0.02825247, -0.15545803, -0.2440..."
6,bank,bank,a slope in the turn of a road or track; the ou...,"[[0.2679793, -0.24696185, 0.35943037, -0.20085..."
7,bank,savings_bank,a container (usually with a slot in the top) f...,"[[-0.25165802, 0.38057274, -0.24903081, -0.251..."
8,bank,bank,a building in which the business of banking tr...,"[[-0.20767291, -0.2030315, -0.027683169, 0.143..."
9,bank,bank,a flight maneuver; aircraft tips laterally abo...,"[[-0.12636508, -0.14194009, 0.4006026, 0.00280..."


## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [75]:
raw_c_lemmas = df_sentences["lemma"].unique()

raw_c_sense_embeddings = retrieve_sense_embeddings(
    raw_c_lemmas,
    get_senses_and_glosses
)

raw_c_sense_embeddings

Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 411.69 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 378.51 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 395.69 examples/s]
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned me

,lemma,sense,gloss,embedding
0,act,act,a legal document codifying the result of delib...,"[[0.19401006, -0.69475263, 0.2513345, -0.01521..."
1,act,act,something that people do or cause to happen,"[[0.24634781, -0.13153115, 0.22377744, -0.1565..."
2,act,act,a subdivision of a play or opera or ballet,"[[-0.044995494, 0.063389756, 0.21340184, 0.000..."
3,act,act,a short theatrical performance that is part of...,"[[0.14058809, -0.00984067, 0.18714945, 0.09102..."
4,act,act,a manifestation of insincerity,"[[0.66068196, -0.40948638, 0.48419464, -0.0957..."
...,...,...,...,...
1418,yard,cubic_yard,a unit of volume (as for sand or gravel),"[[-0.46513122, -0.29797292, 0.0263151, 0.10058..."
1419,yard,yard,a tract of land where logs are accumulated,"[[0.39487115, -0.37448496, 0.120497525, -0.307..."
1420,yard,yard,an area having a network of railway tracks and...,"[[0.60051453, -0.36908296, 0.12968758, -0.4204..."
1421,yard,yard,a long horizontal spar tapered at the end and ...,"[[0.07831305, -0.120901905, 0.46663186, 0.0254..."


In [56]:
raw_c_sense_embeddings.to_pickle("data/processed/raw_c_sense_embeddings.pkl")

In [87]:
raw_c_sense_embeddings

,lemma,sense,gloss,embedding
0,act,act,a legal document codifying the result of delib...,"[[0.19401006, -0.69475263, 0.2513345, -0.01521..."
1,act,act,something that people do or cause to happen,"[[0.24634781, -0.13153115, 0.22377744, -0.1565..."
2,act,act,a subdivision of a play or opera or ballet,"[[-0.044995494, 0.063389756, 0.21340184, 0.000..."
3,act,act,a short theatrical performance that is part of...,"[[0.14058809, -0.00984067, 0.18714945, 0.09102..."
4,act,act,a manifestation of insincerity,"[[0.66068196, -0.40948638, 0.48419464, -0.0957..."
...,...,...,...,...
1418,yard,cubic_yard,a unit of volume (as for sand or gravel),"[[-0.46513122, -0.29797292, 0.0263151, 0.10058..."
1419,yard,yard,a tract of land where logs are accumulated,"[[0.39487115, -0.37448496, 0.120497525, -0.307..."
1420,yard,yard,an area having a network of railway tracks and...,"[[0.60051453, -0.36908296, 0.12968758, -0.4204..."
1421,yard,yard,a long horizontal spar tapered at the end and ...,"[[0.07831305, -0.120901905, 0.46663186, 0.0254..."


# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
import numpy as np
from numpy.linalg import norm
from tqdm import tqdm

df_sentences_copy = df_sentences.copy(deep=True)

matches = {
    "static": {
        "sense": [],
        "gloss": [], 
        "emb": []
    },
    "layer_4": {
        "sense": [],
        "gloss": [], 
        "emb": []
    },
    "layer_8": {
        "sense": [],
        "gloss": [], 
        "emb": []
    },
    "layer_12": {
        "sense": [],
        "gloss": [], 
        "emb": []
    }
}


for _, row in tqdm(df_sentences.iterrows()):
    lemma = row["lemma"]
    v = row["v"]

    sense_embeddings = raw_c_sense_embeddings[raw_c_sense_embeddings["lemma"] == lemma]

    lemmas = sense_embeddings["lemma"]
    senses = sense_embeddings["sense"]
    glosses = sense_embeddings["gloss"]
    
    embeddings = [np.array(emb).flatten() for emb in sense_embeddings["embedding"]]

    for layer_id in ["static", "layer_4", "layer_8", "layer_12"]:
        layer_emb = row[layer_id]

        match_dict = {
            np.dot(layer_emb, embeddings[i]) / (norm(layer_emb) * norm(embeddings[i])): i for i in range(len(embeddings))
        }

        emb_match_idx = sorted(match_dict.items(), key=lambda x: x[0], reverse=True)[0][1]

        matches[layer_id]["sense"].append(senses.iloc[emb_match_idx])
        matches[layer_id]["gloss"].append(glosses.iloc[emb_match_idx])
        matches[layer_id]["emb"].append(embeddings[emb_match_idx])

for layer_id in matches.keys():
    for match_type in matches[layer_id].keys():
        df_sentences_copy[f"{layer_id}_{match_type}_match"] = matches[layer_id][match_type]

reordered_columns = ["lemma", "word", "sentence", "v"] + [
    col
    for layer_id in ["static", "layer_4", "layer_8", "layer_12"]
    for col in [layer_id, f"{layer_id}_sense_match", f"{layer_id}_gloss_match", f"{layer_id}_emb_match"]
]

df_sentences_copy = df_sentences_copy[reordered_columns]

df_sentences_copy.to_pickle("data/processed/raw_c_with_emb_matches.pkl")

df_sentences_copy[df_sentences_copy["lemma"] == "bank"]

448it [00:00, 2197.68it/s]


,lemma,word,sentence,v,static,static_sense_match,static_gloss_match,static_emb_match,layer_4,layer_4_sense_match,layer_4_gloss_match,layer_4_emb_match,layer_8,layer_8_sense_match,layer_8_gloss_match,layer_8_emb_match,layer_12,layer_12_sense_match,layer_12_gloss_match,layer_12_emb_match
15,bank,banked,He banked the plane.,M1_a,"[-0.13131805, 0.0244898, 0.008641114, 0.002749...",bank,do business with a bank or keep an account at ...,"[-0.36580104, -0.05396232, 0.08979216, -0.1198...","[-1.4752624, -0.642939, 0.22964697, 0.35843378...",bank,a supply or stock held in reserve for future u...,"[-0.52822095, 0.034207292, 0.040675588, -0.272...","[0.35498816, -0.78141624, -0.23700708, 0.39223...",bank,cover with ashes so to control the rate of bur...,"[-0.37382233, -0.04510562, -0.0364946, 0.00770...","[0.21270442, -0.08092681, 0.2465183, 0.3262613...",bank,a long ridge or pile,"[-0.020879323, 0.34919986, -0.10445713, 0.0757..."
16,bank,banked,He banked the helicopter.,M1_b,"[-0.13131805, 0.0244898, 0.008641114, 0.002749...",bank,do business with a bank or keep an account at ...,"[-0.36580104, -0.05396232, 0.08979216, -0.1198...","[-1.589306, -0.82592946, 0.098379806, 0.260304...",bank,a long ridge or pile,"[-0.020879323, 0.34919986, -0.10445713, 0.0757...","[0.38711834, -1.0598534, -0.3403873, 0.2529141...",bank,cover with ashes so to control the rate of bur...,"[-0.37382233, -0.04510562, -0.0364946, 0.00770...","[0.35628334, -0.2379035, 0.36186323, 0.1923540...",bank,a long ridge or pile,"[-0.020879323, 0.34919986, -0.10445713, 0.0757..."
17,bank,banked,He banked the money.,M2_a,"[-0.13131805, 0.0244898, 0.008641114, 0.002749...",bank,do business with a bank or keep an account at ...,"[-0.36580104, -0.05396232, 0.08979216, -0.1198...","[-1.6545007, 0.09614863, 1.2335153, 0.79005235...",bank,the funds held by a gambling house or the deal...,"[-0.4247744, 0.02825247, -0.15545803, -0.24408...","[0.11559829, 0.45370638, 0.062498335, 0.081394...",bank,the funds held by a gambling house or the deal...,"[-0.4247744, 0.02825247, -0.15545803, -0.24408...","[0.012232253, -0.017408792, 0.19439478, -0.181...",bank,the funds held by a gambling house or the deal...,"[-0.4247744, 0.02825247, -0.15545803, -0.24408..."
341,bank,banked,He banked the cash.,M2_b,"[-0.13131805, 0.0244898, 0.008641114, 0.002749...",bank,do business with a bank or keep an account at ...,"[-0.36580104, -0.05396232, 0.08979216, -0.1198...","[-1.463408, -0.03926093, 1.3621032, 0.82840425...",bank,the funds held by a gambling house or the deal...,"[-0.4247744, 0.02825247, -0.15545803, -0.24408...","[0.21141349, 0.36641037, 0.1393552, -0.0428510...",bank,the funds held by a gambling house or the deal...,"[-0.4247744, 0.02825247, -0.15545803, -0.24408...","[0.026908025, -0.06287225, 0.30714938, -0.1473...",bank,act as the banker in a game or in gambling,"[-0.41467115, -0.15904076, 0.15017356, -0.0587..."


## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [ ]:
# your code here

Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [ ]:
# your code here

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [ ]:
# your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [ ]:
# your code here

Now, print the matches and mismatches for the static layer only.

In [ ]:
# your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
# your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
# your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
# your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



